# Smart Agriculture Data Analytics using Apache Spark

This notebook processes the India district-level Area, Production and Yield (APY) dataset (approximately 53.62 MiB as listed by India Data Portal). It performs data cleaning, crop/state/season/year aggregations, and production regression using Spark MLlib.

**Dataset source:** Ministry of Agriculture & Farmers Welfare, Government of India, via [India Data Portal](https://indiadataportal.com/p/area-production-yield-apy). Dataset coverage: 1997–2023, district-level. The notebook attempts to fetch the CSV automatically and offers a manual upload fallback if the portal blocks automated access.

**Execution note:** This notebook uses Spark in Colab local mode; it is not a multi-node cluster deployment.

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q install pyspark==3.5.1

import os, subprocess
java_home = subprocess.check_output('dirname $(dirname $(readlink -f $(which java)))', shell=True, text=True).strip()
os.environ['JAVA_HOME'] = java_home
print('JAVA_HOME:', java_home)
print('PySpark is ready')

In [ ]:
!git clone https://github.com/deecares/Smart-Agriculture-Crop-Recommendation.git
%cd Smart-Agriculture-Crop-Recommendation
!mkdir -p data

## Get the large APY dataset

The notebook first attempts to download the CSV directly from the India Data Portal. If the portal blocks automated downloads, it opens a file picker as a fallback.

In [ ]:
import os, requests
from google.colab import files
target_path = 'data/APY.csv'
download_url = 'https://ckandev.indiadataportal.com/dataset/f2bbc28c-6c7c-462b-9064-ea4c4213d466/resource/f980409d-49a2-42ae-9eb0-182365005c04/download/crop-wise-area-production-yield.csv'
downloaded = False
try:
    response = requests.get(download_url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=120, stream=True)
    response.raise_for_status()
    total = 0
    with open(target_path, 'wb') as output:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output.write(chunk)
                total += len(chunk)
    if total > 5 * 1024 * 1024:
        downloaded = True
        print('Downloaded APY dataset directly from India Data Portal.')
    else:
        os.remove(target_path)
        print('The portal response was unexpectedly small; switching to manual upload.')
except Exception as error:
    if os.path.exists(target_path):
        os.remove(target_path)
    print('Automatic download was blocked or unavailable:', error)
if not downloaded:
    print('Please select the APY CSV downloaded from the portal.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No APY CSV was uploaded.')
    uploaded_name = next(iter(uploaded))
    os.replace(uploaded_name, target_path)
print('Saved:', target_path)
print('File size (MiB):', round(os.path.getsize(target_path) / (1024**2), 2))
if os.path.getsize(target_path) < 5 * 1024 * 1024:
    raise ValueError('This does not look like the expected large APY CSV. Check the downloaded file.')

In [ ]:
import pandas as pd
preview = pd.read_csv('data/APY.csv', nrows=5)
print('Columns:', list(preview.columns))
display(preview)

In [ ]:
!spark-submit src/agriculture_analytics.py --input data/APY.csv --output outputs

In [ ]:
import glob
metric_files = glob.glob('outputs/model_metrics/part-*.csv')
if not metric_files:
    raise FileNotFoundError('Model metrics were not created; check the Spark output above.')
metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
display(metrics.sort_values('rmse'))
print('Analytics output folders:', sorted(os.listdir('outputs')))